# Brick 2 — Single-agent Gymnasium environment

We wrap `ecology.step()` (Brick 1) in a standard Gymnasium interface.

**Objectives**:
1. Verify that the Gymnasium API works (`reset` / `step`)
2. Simulate a **random** fisher over one episode
3. Compare several **scripted policies** (passive, moderate, greedy)
4. See the effect of the **physical cap** when the agent demands more than the stock

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from bilevel_fishery.ecology import EcologyParams
from bilevel_fishery.envs import FisheryEnv

plt.rcParams["figure.figsize"] = (10, 5)

## 1. Sanity check: reset + step

We verify the API contract: `reset(seed)` returns `(obs, info)`,
`step(action)` returns `(obs, reward, terminated, truncated, info)`.

In [ ]:
env = FisheryEnv(horizon=200)
obs, info = env.reset(seed=42)
print(f"obs shape={obs.shape}, dtype={obs.dtype}")
print(f"obs (fish_norm, algae_norm) = ({obs[0]:.3f}, {obs[1]:.3f})")
print(f"action_space = {env.action_space}")
print(f"observation_space = {env.observation_space}")

next_obs, reward, terminated, truncated, info = env.step(
    np.array([0.5], dtype=np.float32)
)
print("\nAfter one step at action=0.5:")
print(f"  next obs = ({next_obs[0]:.3f}, {next_obs[1]:.3f})")
print(f"  reward   = {reward:.4f}")
print(f"  terminated = {terminated}, truncated = {truncated}")
print(f"  info     = {info}")

## 2. Random fisher over one episode

Uniform policy on `[0, 1]`. We plot biomasses, realized harvest, and the
cumulative reward.

In [ ]:
env = FisheryEnv(horizon=200)
_, _ = env.reset(seed=42)
rng = np.random.default_rng(42)

fish_history = []
algae_history = []
harvests = []
rewards = []
for _ in range(200):
    action = rng.uniform(0.0, 1.0, size=(1,)).astype(np.float32)
    obs, reward, terminated, truncated, info = env.step(action)
    fish_history.append(info["fish"])
    algae_history.append(info["algae"])
    harvests.append(info["harvest_realized"])
    rewards.append(reward)
    if terminated or truncated:
        break

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(fish_history, label="fish")
axes[0].plot(algae_history, label="algae")
axes[0].set_ylabel("biomass")
axes[0].legend()
axes[0].set_title("Random fisher policy")

axes[1].plot(harvests, color="C2")
axes[1].set_ylabel("harvest_realized")

axes[2].plot(np.cumsum(rewards), color="C3")
axes[2].set_ylabel("cumulative reward")
axes[2].set_xlabel("step")
plt.tight_layout()
plt.show()

print(f"Total reward over episode: {sum(rewards):.4f}")

## 3. Three scripted policies (passive / moderate / greedy)

We compare the total reward and the final stock state for 3 fixed
strategies. Spoiler: the greedy one destroys its own stock.

In [ ]:
policies = {
    "passive (a=0.05)": 0.05,
    "moderate (a=0.25)": 0.25,
    "greedy (a=0.95)": 0.95,
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (name, a) in zip(axes, policies.items(), strict=True):
    env = FisheryEnv(horizon=200)
    env.reset(seed=42)
    action = np.array([a], dtype=np.float32)
    fish_history = []
    total_reward = 0.0
    for _ in range(200):
        _, reward, _, truncated, info = env.step(action)
        fish_history.append(info["fish"])
        total_reward += reward
        if truncated:
            break
    ax.plot(fish_history)
    ax.set_title(f"{name}\nΣreward = {total_reward:.2f}")
    ax.set_xlabel("step")
axes[0].set_ylabel("fish biomass")
plt.tight_layout()
plt.show()

## 4. The physical cap in action

If we demand `action=1.0` (max) with a very low initial stock, the env
**caps** the realized harvest to what is physically available. No crash,
just a lower reward.

In [ ]:
params = EcologyParams(fish_init=0.5, algae_init=10.0, noise_std=0.0)
env = FisheryEnv(params=params, max_harvest_rate=5.0, horizon=50)
env.reset(seed=42)

demanded_traj = []
realized_traj = []
fish_traj = []
for _ in range(50):
    _, _, _, truncated, info = env.step(np.array([1.0], dtype=np.float32))
    demanded_traj.append(info["harvest_demanded"])
    realized_traj.append(info["harvest_realized"])
    fish_traj.append(info["fish"])
    if truncated:
        break

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(demanded_traj, label="demanded", lw=2)
ax1.plot(realized_traj, label="realized (capped)", lw=2)
ax1.set_xlabel("step")
ax1.set_ylabel("harvest")
ax1.set_title("Demand vs realized (action=1.0)")
ax1.legend()

ax2.plot(fish_traj, color="C2")
ax2.set_xlabel("step")
ax2.set_ylabel("fish biomass")
ax2.set_title("Stock dynamics under saturated demand")
plt.tight_layout()
plt.show()

## 5. Takeaways

- `FisheryEnv` honours the Gymnasium 1.x API: `reset → (obs, info)`,
  `step → (obs, reward, terminated, truncated, info)`.
- The action is continuous in `[0, 1]`; the reward is `log(1 + harvest)`,
  concave (see [D-001](../docs/decisions/D-001-reward-function.md)).
- The physical cap avoids any numerical crash while staying interpretable:
  no "phantom fishing".
- Without regulation, a **greedy** fisher eventually collapses the stock —
  this is exactly the *tragedy of the commons* that we'll address with a
  mechanism in **Brick 3**.